# FIFA World Cup 2026 Predictor

Exploration, model evaluation, and tournament simulation results.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.features import FEATURE_COLUMNS, build_feature_dataset
from src.model import train_and_evaluate
from src.preprocessing import configure_logging, load_and_preprocess
from src.simulate import plot_bracket_simulation, plot_monte_carlo_winners, run_monte_carlo

configure_logging()
plt.style.use('dark_background')

## Data Overview

In [ ]:
results, elo = load_and_preprocess(PROJECT_ROOT / 'data' / 'raw')
overview = pd.DataFrame({
    'dataset': ['results', 'elo'],
    'rows': [len(results), len(elo)],
    'columns': [results.shape[1], elo.shape[1]],
    'missing_values': [int(results.isna().sum().sum()), int(elo.isna().sum().sum())],
    'date_min': [results['date'].min(), elo['date'].min()],
    'date_max': [results['date'].max(), elo['date'].max()],
})
display(overview)
display(results.dtypes.to_frame('results_dtype'))
display(elo.dtypes.to_frame('elo_dtype'))

## ELO Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(elo['elo_rating'], bins=40, color='#00FF87', ax=ax)
ax.set_title('ELO Rating Distribution')
ax.set_xlabel('ELO rating')
plt.show()

top_elo = elo.sort_values('date').groupby('team').tail(1).sort_values('elo_rating', ascending=False).head(10)
display(top_elo[['team', 'elo_rating', 'date']])

## Feature Correlations

In [ ]:
features = build_feature_dataset(results, elo)
display(features.head())
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(features[FEATURE_COLUMNS].corr(), cmap='Greens', center=0, ax=ax)
ax.set_title('Feature Correlations')
plt.show()

## Model Training & Evaluation

In [ ]:
outputs_dir = PROJECT_ROOT / 'outputs'
visualizations_dir = PROJECT_ROOT / 'visualizations'
model, label_encoder, metrics = train_and_evaluate(features, outputs_dir, visualizations_dir)
display(pd.DataFrame(metrics['classification_report']).T)
display(pd.DataFrame(metrics['confusion_matrix'], index=metrics['classes'], columns=metrics['classes']))

## Simulation Results

In [ ]:
summary, round_counts, context = run_monte_carlo(model, label_encoder, results, elo, simulations=1000)
display(summary.head(20))
plot_monte_carlo_winners(summary, visualizations_dir / 'notebook_monte_carlo_winners.png')
display(summary.head(20)[['team', 'win_probability', 'final_probability', 'sf_probability']])

## Most Likely Bracket

In [ ]:
plot_bracket_simulation(summary, round_counts, 1000, visualizations_dir / 'notebook_bracket_simulation.png')
img = plt.imread(visualizations_dir / 'notebook_bracket_simulation.png')
fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img)
ax.axis('off')
plt.show()